# 실습 1. YOLO 객체 탐지 기초


# Google Colab 실행 안내
이 교안은 **짧은 단계별 실습**으로 구성되어 있습니다. Markdown 설명을 읽고 바로 아래 코드 셀을 하나씩 실행하세요.

- 실행 환경: Google Colab, 인터넷 연결, MP4 동영상 파일
- 프레임(frame): 동영상을 이루는 한 장의 이미지
- 객체 탐지: 이미지에서 객체의 종류와 위치를 찾는 작업
- 바운딩 박스: 탐지 객체를 둘러싼 사각형
- 신뢰도(confidence): 모델의 탐지 점수
- 추론(inference): 학습된 모델로 새 이미지의 객체를 찾는 과정

유튜브 영상은 이용약관과 저작권을 준수해 사용하세요. 수업에서는 직접 촬영하거나 공개 라이선스/사용 허가를 받은 영상을 권장합니다. 이 노트북은 영상 파일을 직접 업로드합니다. 처음에는 10~30초 길이의 짧은 영상으로 시작하세요.


## 실습 목표
- 이미지에서 객체를 탐지하고 이름·신뢰도·좌표를 읽습니다.
- 신뢰도 기준을 바꾸어 결과를 비교합니다.
- 프레임별 탐지 수를 CSV와 그래프로 정리합니다.

## 실습 시나리오
동영상에서 일정 간격으로 프레임을 추출해 일반 객체를 탐지합니다.

## 1단계. 라이브러리 설치
`!`는 Colab 셀에서 운영체제 명령을 실행하라는 뜻입니다. `ultralytics`는 YOLO 모델, `opencv-python`은 영상 읽기, `pandas`는 표 데이터, `matplotlib`은 그래프에 사용합니다.

In [ ]:
!pip -q install ultralytics opencv-python pandas matplotlib

## 2단계. 분석할 동영상 업로드
파일 선택 창에서 MP4 동영상을 선택합니다. `uploaded`는 업로드한 파일 정보를 담고, `next(iter(uploaded))`는 첫 번째 파일 이름을 가져옵니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))
print("업로드한 영상:", video_path)

## 3단계. YOLO 모델 불러오기
`YOLO("yolo11n.pt")`는 가벼운 사전 학습 모델을 불러옵니다. 처음에는 모델 파일 다운로드가 필요할 수 있습니다. 이 모델은 일반 객체 탐지용이며, 특정 안전모나 작물 질병을 자동으로 정확히 판정하는 전문 모델은 아닙니다.

In [ ]:
from ultralytics import YOLO
import cv2, pandas as pd, matplotlib.pyplot as plt
model = YOLO("yolo11n.pt")
print("모델 준비 완료")

## 4단계. 영상에서 첫 프레임 읽기
`VideoCapture()`로 영상을 열고 `read()`로 이미지 한 장을 읽습니다. `ret`이 True면 성공, `frame`은 이미지 배열입니다. OpenCV의 색상 순서는 BGR이므로 Matplotlib에 표시할 때 RGB로 변환합니다.

In [ ]:
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()
if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(frame_rgb)
    plt.axis("off")
    plt.show()
else:
    print("영상을 읽지 못했습니다. 파일을 확인하세요.")

## 5단계. 첫 프레임에서 객체 탐지
`model(frame)`은 이미지에서 객체를 찾습니다. `results[0]`은 첫 이미지의 결과, `plot()`은 상자와 클래스 이름을 그린 이미지를 반환합니다.

In [ ]:
results = model(frame, verbose=False)
annotated = results[0].plot()
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
print("탐지 객체 수:", len(results[0].boxes))

## 6단계. 객체 이름·신뢰도·좌표 출력
`for`는 탐지 결과를 하나씩 처리합니다. `box.cls[0]`은 클래스 번호, `model.names[...]`는 클래스 이름, `box.conf[0]`은 신뢰도, `box.xyxy[0]`은 좌표 `(x1,y1,x2,y2)`입니다. `int()`와 `float()`는 숫자 형식을 바꿉니다.

In [ ]:
for box in results[0].boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = [round(float(v), 1) for v in box.xyxy[0]]
    print(f"객체={model.names[class_id]}, 신뢰도={confidence:.2f}, 좌표=({x1}, {y1}, {x2}, {y2})")

## 7단계. 신뢰도 기준 비교
`conf`는 최소 신뢰도 기준입니다. 0.25, 0.50, 0.75로 바꾸어 보세요. 기준을 높이면 낮은 점수의 탐지가 사라지지만 실제 객체도 놓칠 수 있습니다.

In [ ]:
for threshold in [0.25, 0.50, 0.75]:
    res = model(frame, conf=threshold, verbose=False)[0]
    print(f"기준={threshold:.2f} / 탐지 수={len(res.boxes)}")

## 8단계. 짧은 구간의 프레임별 탐지 수 분석
모든 프레임을 분석하면 오래 걸릴 수 있으므로 일정 간격으로 분석합니다. `frame_step=30`은 30프레임마다 한 번입니다. 영상이 30 FPS라면 약 1초 간격입니다.

In [ ]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_step = 30
rows = []; idx = 0
while True:
    ret, img = cap.read()
    if not ret: break
    if idx % frame_step == 0:
        res = model(img, verbose=False)[0]
        rows.append({"frame": idx, "seconds": idx/fps, "detections": len(res.boxes)})
    idx += 1
cap.release()
df = pd.DataFrame(rows)
display(df.head(10))

## 9단계. 그래프와 CSV 저장
프레임별 탐지 수를 그래프로 그리고 CSV로 저장합니다. 주의: 이 수치는 프레임마다 보인 객체 수이지, 영상 전체의 고유 인원 수가 아닙니다.

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df["seconds"], df["detections"], marker="o")
plt.xlabel("시간(초)"); plt.ylabel("프레임별 탐지 수")
plt.title("시간에 따른 객체 탐지 수"); plt.grid(True); plt.show()
df.to_csv("object_detection_log.csv", index=False)
files.download("object_detection_log.csv")

## 학생 과제 및 제출물
1. 신뢰도 기준 3개 결과를 비교합니다.
2. 잘 탐지한 객체 3개와 놓친 객체 2개를 기록합니다.
3. **제출물:** 탐지 이미지, CSV, 결과 해석 5문장. 모델이 객체를 놓치거나 잘못 분류할 수 있는 이유도 적으세요.